In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import traceback
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient
import pandas as pd

f:\DiskE\SKRIPSI-CODE\skripsi-code\ai\.venv\Lib\site-packages\mlflow\utils\autologging_utils\versioning.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


### Try to load using MLflow API

This doesn't work, scroll below to the next chapter

In [ ]:
load_dotenv()

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
mlflow.set_tracking_uri(tracking_uri)  # type: ignore
client: MlflowClient = MlflowClient(tracking_uri=tracking_uri)

# RUN_ID = "906cef2b2f4f465eb47f298a50065e54"
RUN_ID = "15fc890603d149e4918a1c6cfbf9aa48"

METRICS_TO_PLOT: list[str] = [
    # "avg_train_loss",
    "val_loss_avg",
    # "val_macro_pr_auc",
    # "val_macro_roc_auc",
]

run = client.get_run(RUN_ID)
print(list(run.data.metrics.keys()))
# print(run)


def get_and_clean_metric(client: MlflowClient, run_id, metric_key):
    """Fetches metric history from MLflow and removes NaN or None values."""
    try:
        history = client.get_metric_history(run_id, metric_key)
        print("history", history)

        steps = []
        values = []

        for m in history:
            if m.value is not None and not np.isnan(m.value):
                steps.append(m.step)
                values.append(m.value)

        return steps, values

    except Exception as e:
        traceback.print_exc()
        print(f"Could not retrieve '{metric_key}': {e}")
        return [], []


num_metrics = len(METRICS_TO_PLOT)
fig, axes = plt.subplots(nrows=num_metrics, ncols=1, figsize=(10, 4 * num_metrics))

# Check axes is iterable even if there's only one metric
if num_metrics == 1:
    axes = [axes]

for ax, metric_key in zip(axes, METRICS_TO_PLOT):
    steps, values = get_and_clean_metric(client, RUN_ID, metric_key)

    if len(steps) == 0:
        ax.text(
            0.5,
            0.5,
            f"No valid data for {metric_key}",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title(metric_key)
        continue

    ax.plot(steps, values, marker="o", linestyle="-", color="b", label=metric_key)
    ax.set_title(f"{metric_key} over Epochs")
    ax.set_xlabel("Epoch / Step")
    ax.set_ylabel("Value")
    ax.grid(True, linestyle="--", alpha=1)
    ax.legend()

plt.tight_layout()
plt.show()

# This works

In [ ]:
df = pd.read_csv("./mlflow_pgdump/906cef_history.csv")
df.head()

In [ ]:
def plot_db_metrics(df: pd.DataFrame, metrics_to_plot: list[str]):
    """
    Takes a raw MLflow PostgreSQL metrics dump and a list of metric keys,
    filters out the NaNs, and plots them in stacked subplots.
    """
    num_metrics = len(metrics_to_plot)

    if num_metrics == 0:
        print("No metrics provided to plot.")
        return

    _, axes = plt.subplots(nrows=num_metrics, ncols=1, figsize=(10, 4 * num_metrics))

    if num_metrics == 1:
        axes = [axes]

    for ax, metric_key in zip(axes, metrics_to_plot):
        if "pr" in metric_key:
            df_clean = df[
                (df["key"] == metric_key)
                & (df["is_nan"].isin(["f", "F", False, 0]))
                & (df["value"] > 0.0)
            ].copy()
        elif "roc" in metric_key:
            df_clean = df[
                (df["key"] == metric_key)
                & (df["is_nan"].isin(["f", "F", False, 0]))
                & (df["value"] > 0.5)
            ].copy()
        else:
            df_clean = df[
                (df["key"] == metric_key)
                & (df["is_nan"].isin(["f", "F", False, 0]))
                & (df["value"].notna())
            ].copy()

        df_clean = df_clean.sort_values(by="step")

        # Handle edge case where the metric exists but all values were NaN
        if df_clean.empty:
            ax.text(
                0.5,
                0.5,
                f"No valid data found for '{metric_key}'",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_title(metric_key)
            continue

        ax.plot(
            df_clean["step"].apply(lambda x: x + 1),
            df_clean["value"],
            marker="o",
            linestyle="-",
            color="b",
            label=metric_key,
        )

        ax.set_title(f"{metric_key} over epochs")
        ax.set_xlabel("epoch")
        ax.set_xticks(np.arange(1, stop=len(df_clean["step"]) + 1))
        ax.set_ylabel("value")
        ax.grid(True, linestyle="--", alpha=0.7)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
METRICS_TO_PLOT = [
    "train_loss",
    "avg_train_loss",
    "val_loss_avg",
    "val_macro_pr_auc",
    "val_macro_roc_auc"
]

plot_db_metrics(df, METRICS_TO_PLOT)

# Testing results

In [31]:
adf = pd.read_csv("./mlflow_pgdump/9f3cb9_history.csv")
adf.head()

,key,value,timestamp,run_uuid,step,is_nan
0,test_loss_avg,0.054008,1780424812190,9f3cb99c242f4307a6f8872c2f73f11b,0,f
1,test_loss_std,0.024468,1780424812190,9f3cb99c242f4307a6f8872c2f73f11b,0,f
2,test_pr_auc_mood_theme---action,0.032234,1780424812190,9f3cb99c242f4307a6f8872c2f73f11b,0,f
3,test_roc_auc_mood_theme---action,0.694500,1780424812190,9f3cb99c242f4307a6f8872c2f73f11b,0,f
4,test_precision_mood_theme---action,0.047619,1780424812190,9f3cb99c242f4307a6f8872c2f73f11b,0,f


In [29]:
print(adf[(adf["step"] == 0) & (adf["key"] == "test_macro_pr_auc")])
print(adf[(adf["step"] == 0) & (adf["key"] == "test_macro_roc_auc")])
# print(adf[(adf["step"] == 24) & (adf["key"] == "test_macro_pr_auc")])
# print(adf[(adf["step"] == 24) & (adf["key"] == "test_macro_roc_auc")])

# print(adf[adf["step"] == 24])

exclusion = [
    "test_macro_pr_auc",
    "test_macro_roc_auc",
    "test_loss_avg",
    "test_loss_std",
]

prauc_keys = []
prauc_vals = []
rocauc_keys = []
rocauc_vals = []
optimal_keys = []
optimal_vals = []

for item in adf.iterrows():
    # print(item[1]["key"])
    if item[1]["key"] not in exclusion:
        clean_key = "_".join(item[1]["key"].split("_")[1:]).replace("mood/theme---", "")
        optimal_key = "_".join(item[1]["key"].split("_")).replace("mood/theme---", "")
        if "pr" in clean_key and "precision" not in clean_key:
            prauc_keys.append(clean_key)
            prauc_vals.append(item[1]["value"])
        elif "roc" in clean_key:
            rocauc_keys.append(clean_key)
            rocauc_vals.append(item[1]["value"])
        elif "optimal" in optimal_key:
            optimal_keys.append(optimal_key)
            optimal_vals.append(item[1]["value"])

        # print(f"{clean_key} {item[1]["value"]}")

print(len(prauc_keys))
print(len(prauc_vals))
print(len(rocauc_keys))
print(len(rocauc_vals))
print(len(optimal_keys))
print(len(optimal_vals))

                   key     value      timestamp  \
282  test_macro_pr_auc  0.125456  1780425884802   

                             run_uuid  step is_nan  
282  a9889184136f49978b22e36ed09fb0ba     0      f  
                    key     value      timestamp  \
283  test_macro_roc_auc  0.744919  1780425884802   

                             run_uuid  step is_nan  
283  a9889184136f49978b22e36ed09fb0ba     0      f  
56
56
56
56
56
56


In [ ]:
# for i, item in enumerate(prauc_keys):
    # print(f"{str(item).replace("pr_auc_", "")} {prauc_vals[i]}")
    # print(f"{str(item).replace("pr_auc_", "")}")

In [19]:
for item in prauc_vals:
    print(item)

0.0280303262157614
0.0713221358170316
0.1819177945066687
0.0197566988050875
0.0748754491331882
0.0361563494191222
0.3048208451386711
0.1911543149522212
0.1326870051996215
0.0268835447489858
0.2244950812509181
0.2823487077144743
0.6562388519173918
0.0597913055548784
0.0397602847694211
0.0447738079994625
0.0786913432279588
0.1471987339952471
0.1407182943197202
0.3746796409245902
0.0499261590305774
0.3010330469007902
0.1204681374244398
0.0814210969907742
0.0323595314464453
0.0363312389283354
0.2683962884976222
0.1630698977781424
0.0374048849309252
0.0136841573610634
0.0902572760741566
0.1516389784667552
0.1609085391256757
0.0378051233298155
0.0970262777472637
0.2466094408742564
0.0241931441934302
0.023434504913982
0.0655999110139204
0.0760138517389178
0.1026938760947674
0.1526285551757873
0.0115911424771133
0.1116860999353645
0.1067509835288198
0.0185008729540923
0.0452975274464839
0.0661973323453977
0.0385050622764211
0.0607690457667405
0.0794582455685105
0.531149545402034
0.144281130393

In [20]:
for item in rocauc_vals:
    print(item)

0.7036954662104363
0.6888885077186964
0.7941859553239107
0.5985575001997923
0.7802453788151642
0.6923724296574747
0.8323723129951504
0.7826164654525612
0.8127696378895751
0.7133253325739393
0.8544790385748091
0.8098716480159779
0.9362356876183412
0.6925892346647063
0.6956537260151717
0.7184928576130605
0.635643316909957
0.6388655249930539
0.7433908925649195
0.8496878918800704
0.8432949308755759
0.7631404660412981
0.8344880639401188
0.8101774106175514
0.6567608861726508
0.7962520888040105
0.7613464160372813
0.9195149786019972
0.7262610781635237
0.5816230203682726
0.622999808003511
0.7392368668351373
0.8153662565668789
0.6610203296625019
0.6692935710453148
0.7968935012445992
0.513483735844546
0.7021242613001117
0.8598463079066526
0.7492336237495966
0.8104565704718936
0.7218495285906901
0.6445637893946612
0.7760064385238342
0.7040833615723484
0.7367785603219611
0.7304564369381344
0.7283344483122242
0.8301301514877973
0.8140688501075783
0.8348806366047745
0.8756695419707854
0.9056012884753

In [32]:
for item in optimal_vals:
    print(item)

0.8735490441322327
0.8224253058433533
0.6875119805335999
0.6172891855239868
0.9057767987251282
0.8412103056907654
0.7187037467956543
0.8854371905326843
0.8271136283874512
0.6930103898048401
0.8282457590103149
0.6200806498527527
0.8883222937583923
0.6295610666275024
0.8409250974655151
0.7448381185531616
0.7638742327690125
0.7745650410652161
0.6688358187675476
0.8738225102424622
0.8494932055473328
0.800399661064148
0.5285919904708862
0.8530703783035278
0.8746718764305115
0.849007785320282
0.5903968811035156
0.7970151305198669
0.9193935394287108
0.795075535774231
0.7724539041519165
0.7658712267875671
0.857116162776947
0.6759597063064575
0.5733886957168579
0.8721718788146973
0.7887535095214844
0.7471622824668884
0.8195759654045105
0.5927609801292419
0.7447736263275146
0.8166205883026123
0.4308451116085052
0.8084613084793091
0.8182446360588074
0.5139854550361633
0.7457034587860107
0.7301368117332458
0.8486478924751282
0.643641471862793
0.8775589466094971
0.8277892470359802
0.771963536739349

# PR/ROC curves

In [7]:
import os
import glob
import math
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.artifacts
from dotenv import load_dotenv

# Folder to save the output plots locally
OUTPUT_PLOTS_DIR = "./plots"

def load_environment():
    load_dotenv()
    mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI")
    if not mlflow_uri:
        raise ValueError("MLFLOW_TRACKING_URI not found in environment variables.")
    mlflow.set_tracking_uri(mlflow_uri)
    print(f"Tracking URI set to: {mlflow_uri}")

def extract_label_from_filename(filepath, prefix):
    """
    Extracts the 'safe_label' from a filename like 'pr_curve_Acoustic_Guitar.csv'.
    """
    filename = os.path.basename(filepath)
    label = filename.replace(prefix, "").replace(".csv", "")
    return label

def plot_grid(files, prefix, x_col, y_col, plot_title, output_filename, is_roc=False):
    """
    Plots a grid of curves. Defaults to 4 columns and calculates rows dynamically.
    """
    if not files:
        print(f"No files found for {plot_title}. Skipping.")
        return

    # Sort files alphabetically by the label name
    files = sorted(files, key=lambda f: extract_label_from_filename(f, prefix))
    
    num_plots = len(files)
    cols = 4
    rows = math.ceil(num_plots / cols)

    # Make the figure very large so 56 subplots are readable
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(6 * cols, 4 * rows))
    fig.suptitle(plot_title, fontsize=24, y=0.99)
    
    axes = axes.flatten()

    for i, filepath in enumerate(files):
        ax = axes[i]
        label_name = extract_label_from_filename(filepath, prefix)
        
        try:
            df = pd.read_csv(filepath)
            
            # Plot the actual curve
            ax.plot(df[x_col], df[y_col], color='b', lw=2)
            
            # If ROC, add the diagonal chance line
            if is_roc:
                ax.plot([0, 1], [0, 1], color='gray', linestyle='--')
                
            ax.set_title(label_name, fontsize=14, weight='bold')
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xlabel(x_col.upper())
            ax.set_ylabel(y_col.capitalize())
            ax.grid(True, linestyle='--', alpha=0.6)
            
        except Exception as e:
            ax.set_title(f"{label_name} (Error)", fontsize=14)
            ax.text(0.5, 0.5, str(e), ha='center', va='center', fontsize=8)

    # Hide any unused subplots if the number of files isn't a perfect multiple of 4
    for j in range(num_plots, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout(pad=3.0)
    
    out_path = os.path.join(OUTPUT_PLOTS_DIR, output_filename)
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f"Saved grid plot to: {out_path}")
    plt.close(fig)

def plot_macro(filepath, x_col, y_col, plot_title, output_filename, is_roc=False):
    """
    Plots a single macro-average curve.
    """
    if not os.path.exists(filepath):
        print(f"Macro file not found: {filepath}. Skipping.")
        return
        
    try:
        df = pd.read_csv(filepath)
        
        plt.figure(figsize=(8, 6))
        plt.plot(df[x_col], df[y_col], color='purple', lw=3, label="Macro Average")
        
        if is_roc:
            plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label="Chance")
            
        plt.title(plot_title, fontsize=16, weight='bold')
        plt.xlim([-0.05, 1.05])
        plt.ylim([-0.05, 1.05])
        plt.xlabel(x_col.upper(), fontsize=12)
        plt.ylabel(y_col.capitalize(), fontsize=12)
        plt.legend(loc="lower right")
        plt.grid(True, linestyle='--', alpha=0.6)
        
        out_path = os.path.join(OUTPUT_PLOTS_DIR, output_filename)
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        print(f"Saved macro plot to: {out_path}")
        plt.close()
        
    except Exception as e:
        print(f"Error plotting {filepath}: {e}")

def plot_all_curves(target_run_id: str):
    load_environment()
    
    if target_run_id == "YOUR_TESTING_RUN_ID_HERE":
        print("Please replace 'target_run_id' in the script with an actual MLflow run ID.")
        return

    os.makedirs(OUTPUT_PLOTS_DIR, exist_ok=True)

    print(f"Downloading artifacts for run {target_run_id}...")
    try:
        local_artifact_dir = mlflow.artifacts.download_artifacts(
            run_id=target_run_id,
            artifact_path="calculated_metrics"
        )
        print(f"Artifacts downloaded to: {local_artifact_dir}")
    except Exception as e:
        print(f"Failed to download artifacts. Are you sure the run ID is correct and artifacts exist? Error: {e}")
        return

    # 1. Gather files
    pr_files = glob.glob(os.path.join(local_artifact_dir, "pr_curve_*.csv"))
    roc_files = glob.glob(os.path.join(local_artifact_dir, "roc_curve_*.csv"))

    # Separate individual label files from the macro average file
    pr_label_files = [f for f in pr_files if "macro_average" not in f]
    roc_label_files = [f for f in roc_files if "macro_average" not in f]
    
    pr_macro_file = os.path.join(local_artifact_dir, "pr_curve_macro_average.csv")
    roc_macro_file = os.path.join(local_artifact_dir, "roc_curve_macro_average.csv")

    print(f"Found {len(pr_label_files)} PR files and {len(roc_label_files)} ROC files for individual labels.")

    # 2. Plot Per-Label Grid: PR-AUC (14x4)
    plot_grid(
        files=pr_label_files,
        prefix="pr_curve_",
        x_col="recall",
        y_col="precision",
        plot_title="Precision-Recall Curves by Label",
        output_filename=f"grid_pr_curves_{target_run_id}.png",
        is_roc=False
    )

    # 3. Plot Per-Label Grid: ROC-AUC (14x4)
    plot_grid(
        files=roc_label_files,
        prefix="roc_curve_",
        x_col="fpr",
        y_col="tpr",
        plot_title="ROC Curves by Label",
        output_filename=f"grid_roc_curves_{target_run_id}.png",
        is_roc=True
    )

    # 4. Plot Macro-Averages
    plot_macro(
        filepath=pr_macro_file,
        x_col="recall",
        y_col="precision",
        plot_title="Macro-Averaged Precision-Recall Curve",
        output_filename=f"macro_pr_curve_{target_run_id}.png",
        is_roc=False
    )
    
    plot_macro(
        filepath=roc_macro_file,
        x_col="fpr",
        y_col="tpr",
        plot_title="Macro-Averaged ROC Curve",
        output_filename=f"macro_roc_curve_{target_run_id}.png",
        is_roc=True
    )
    
    print("\nPlotting complete! Check the './plots' directory for the output images.")

In [9]:
# plot_all_curves("9f3cb99c242f4307a6f8872c2f73f11b") # CNN
# plot_all_curves("5192ad9a81b4484ea3ec447986ac3f9f") # GRU
# plot_all_curves("a9889184136f49978b22e36ed09fb0ba") # ATTN
plot_all_curves("d686499c115d46afb69c591f3ba6863b") # GRU tahap 2

Tracking URI set to: https://mlflow.hugoalfedo.dev
Artifacts downloaded to: C:\Users\guido\AppData\Local\Temp\tmpc1vjbfom\calculated_metrics
Found 56 PR files and 56 ROC files for individual labels.
Saved grid plot to: ./plots\grid_pr_curves_d686499c115d46afb69c591f3ba6863b.png
Saved grid plot to: ./plots\grid_roc_curves_d686499c115d46afb69c591f3ba6863b.png
Saved macro plot to: ./plots\macro_pr_curve_d686499c115d46afb69c591f3ba6863b.png
Saved macro plot to: ./plots\macro_roc_curve_d686499c115d46afb69c591f3ba6863b.png

Plotting complete! Check the './plots' directory for the output images.
